<a href="https://colab.research.google.com/github/Maria-apsguiar/GreenAlpha_Portfolio_Otimizacao-financeira-e-Filtros-de-Carbono-ESG/blob/main/GreenAlpha.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Import de Bibliotecas e Dados

In [ ]:
"""
Case 3: Fintech ESG Portfolio Optimization (Markowitz + Filtros Verdes)
Download de dados reais da B3 via Yahoo Finance + Emissoes GHG Protocol
"""

import os
import sqlite3
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.optimize import minimize

# Fixar semente para reprodutibilidade das simulacoes de Monte Carlo
np.random.seed(42)

# 1. Metadados e Emissoes Oficiais (Programa Brasileiro GHG Protocol / B3 ICO2)
ASSETS_METADATA = [
    # (Ticker, Empresa, Setor, Subsetor, Escopo1_tCO2e, Escopo2_tCO2e, Escopo3_tCO2e, Intensidade_tCO2e_M, Classificacao)
    ("ITUB4.SA", "Itau Unibanco", "Financeiro", "Bancos", 12500, 38000, 2400000, 1.8, "Lider Verde (Baixa Emissao)"),
    ("WEGE3.SA", "WEG", "Bens Industriais", "Motores & Energia Eolica", 85000, 110000, 1850000, 8.4, "Lider Verde (Transicao Energetica)"),
    ("RENT3.SA", "Localiza", "Consumo Ciclico", "Aluguel de Frotas", 95000, 18000, 4200000, 9.2, "Moderada Emissao"),
    ("CPFE3.SA", "CPFL Energia", "Utilidade Publica", "Energia Eletrica / Renovaveis", 240000, 42000, 980000, 14.5, "Moderada Emissao"),
    ("EQTL3.SA", "Equatorial Energia", "Utilidade Publica", "Distribuicao Eletrica", 310000, 55000, 1200000, 18.2, "Moderada Emissao"),
    ("KLBN11.SA", "Klabin", "Materiais Basicos", "Papel & Embalagens", 890000, 120000, 1600000, 38.5, "Intensiva com Remocao"),
    ("SUZB3.SA", "Suzano", "Materiais Basicos", "Papel & Celulose", 2100000, 180000, 3100000, 48.0, "Intensiva com Remocao"),
    ("VBBR3.SA", "Vibra Energia", "Petroleo & Biocombustiveis", "Distribuicao", 450000, 68000, 8500000, 54.0, "Intensiva em Escopo 3"),
    ("VALE3.SA", "Vale", "Materiais Basicos", "Mineracao", 8900000, 540000, 89000000, 88.0, "Alta Intensidade"),
    ("PETR4.SA", "Petrobras", "Petroleo & Gas", "Exploracao & Refino", 52000000, 2100000, 380000000, 112.0, "Altissima Intensidade")
]

def build_esg_fintech_project(output_dir):
    os.makedirs(output_dir, exist_ok=True)

    df_assets = pd.DataFrame(ASSETS_METADATA, columns=[
        "Ticker", "Empresa", "Setor", "Subsetor",
        "Emissoes_Escopo1_tCO2e", "Emissoes_Escopo2_tCO2e", "Emissoes_Escopo3_tCO2e",
        "Intensidade_Carbono_tCO2e_M", "Classificacao_ESG"
    ])

    tickers = df_assets["Ticker"].tolist()
    print("[1/5] Baixando cotacoes historicas reais da B3 via yfinance (2022-2025)...")
    raw_data = yf.download(tickers, start="2022-01-01", end="2025-01-01", progress=False)['Close']

    # Preencher falhas eventuais por forward fill
    price_data = raw_data.ffill().dropna()
    returns = price_data.pct_change().dropna()

    # Retornos anualizados e Matriz de Covariância anualizada (252 dias úteis)
    mean_returns = returns.mean() * 252
    cov_matrix = returns.cov() * 252
    rf = 0.105 # Taxa Selic / CDI livre de risco benchmark (10.5% a.a.)

    intensidades = df_assets.set_index("Ticker").loc[tickers, "Intensidade_Carbono_tCO2e_M"].values

    print("[2/5] Otimizando Carteiras (Markowitz Tradicional vs. Mínima Volatilidade vs. GreenAlpha ESG)...")

    # Funções de Otimização
    def portfolio_performance(weights):
        p_ret = np.dot(weights, mean_returns)
        p_vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
        p_sharpe = (p_ret - rf) / p_vol
        p_carbon = np.dot(weights, intensidades)
        return p_ret, p_vol, p_sharpe, p_carbon

    def min_sharpe_negative(weights):
        return -portfolio_performance(weights)[2]

    def min_variance(weights):
        return portfolio_performance(weights)[1]

    # Limites de Alocação: Nenhum ativo pode ultrapassar 30% da carteira, e sem venda a descoberto (0% a 30%)
    bounds = tuple((0.0, 0.30) for _ in range(len(tickers)))
    init_weights = np.array(len(tickers) * [1.0 / len(tickers)])

    # 1. Carteira Tradicional Max Sharpe (Sem filtro ESG)
    cons_trad = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]
    opt_trad = minimize(min_sharpe_negative, init_weights, method='SLSQP', bounds=bounds, constraints=cons_trad)
    w_trad = opt_trad.x
    ret_trad, vol_trad, sharpe_trad, carb_trad = portfolio_performance(w_trad)

    # 2. Carteira Minima Volatilidade (Mais defensiva)
    opt_min_vol = minimize(min_variance, init_weights, method='SLSQP', bounds=bounds, constraints=cons_trad)
    w_min_vol = opt_min_vol.x
    ret_min_vol, vol_min_vol, sharpe_min_vol, carb_min_vol = portfolio_performance(w_min_vol)

    # 3. Carte opt_green.x
    ret_green, vol_green, sharpe_green, carb_green = portfolio_performance(w_green)

    print("[3/5] Simulando 2.500 carteiras de Monte Carlo para a Fronteira Eficiente...")
    mc_records = []
    for _ in range(25ira GreenAlpha ESG (Max Sharpe com teto de carbono rigoroso: <= 20.0 tCO2e / R$M)
    teto_carbono = 20.0
    cons_green = [
        {'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0},
        {'type': 'ineq', 'fun': lambda w: teto_carbono - np.dot(w, intensidades)}
    ]
    opt_green = minimize(min_sharpe_negative, init_weights, method='SLSQP', bounds=bounds, constraints=cons_green)
    w_green =00):
        w = np.random.random(len(tickers))
        w /= np.sum(w)
        r, v, s, c = portfolio_performance(w)
        mc_records.append({
            "Retorno_Anualizado_%": round(r * 100, 2),
            "Volatilidade_Anualizada_%": round(v * 100, 2),
            "Indice_Sharpe": round(s, 3),
            "Intensidade_Carbono_Carteira": round(c, 2)
        })
    df_monte_carlo = pd.DataFrame(mc_records)

    # Tabela de Pesos por Carteira
    weights_records = []
    for t, wt, wm, wg in zip(tickers, w_trad, w_min_vol, w_green):
        empresa_nome = df_assets.loc[df_assets["Ticker"] == t, "Empresa"].values[0]
        setor_nome = df_assets.loc[df_assets["Ticker"] == t, "Setor"].values[0]
        intens = df_assets.loc[df_assets["Ticker"] == t, "Intensidade_Carbono_tCO2e_M"].values[0]

        weights_records.append({
            "Ticker": t,
            "Empresa": empresa_nome,
            "Setor": setor_nome,
            "Intensidade_Carbono_Ativo": intens,
            "Peso_Carteira_Tradicional_%": round(wt * 100, 2),
            "Peso_Carteira_Min_Vol_%": round(wm * 100, 2),
            "Peso_Carteira_GreenAlpha_ESG_%": round(wg * 100, 2)
        })
    df_weights = pd.DataFrame(weights_records)

    # Resumo Executivo das Carteiras
    df_resumo = pd.DataFrame([
        {
            "Carteira": "Tradicional (Max Sharpe Irrestrito)",
            "Descricao": "Otimizacao pura de retorno ajustado ao risco sem limites de emissoes",
            "Retorno_Anualizado_%": round(ret_trad * 100, 2),
            "Volatilidade_Anualizada_%": round(vol_trad * 100, 2),
            "Indice_Sharpe": round(sharpe_trad, 3),
            "Intensidade_Carbono_tCO2e_M": round(carb_trad, 2),
            "Reducao_Carbono_vs_Tradicional_%": 0.0
        },
        {
            "Carteira": "GreenAlpha ESG (Filtro Verde)",
            "Descricao": "Max Sharpe com teto maximo de 20.0 tCO2e/M de receita",
            "Retorno_Anualizado_%": round(ret_green * 100, 2),
            "Volatilidade_Anualizada_%": round(vol_green * 100, 2),
            "Indice_Sharpe": round(sharpe_green, 3),
            "Intensidade_Carbono_tCO2e_M": round(carb_green, 2),
            "Reducao_Carbono_vs_Tradicional_%": round(((carb_trad - carb_green) / carb_trad) * 100, 2)
        },
        {
            "Carteira": "Minima Volatilidade (Defensiva)",
            "Descricao": "Minimizacao absoluta do desvio padrao da carteira",
            "Retorno_Anualizado_%": round(ret_min_vol * 100, 2),
            "Volatilidade_Anualizada_%": round(vol_min_vol * 100, 2),
            "Indice_Sharpe": round(sharpe_min_vol, 3),
            "Intensidade_Carbono_tCO2e_M": round(carb_min_vol, 2),
            "Reducao_Carbono_vs_Tradicional_%": round(((carb_trad - carb_min_vol) / carb_trad) * 100, 2)
        }
    ])

    # Formatação de Cotações Diárias para Fato de Séries Temporais
    df_cotacoes_melted = price_data.reset_index().melt(id_vars=['Date'], var_name='Ticker', value_name='Preco_Fechamento_Ajustado')
    df_cotacoes_melted['Date'] = pd.to_datetime(df_cotacoes_melted['Date']).dt.strftime('%Y-%m-%d')

    print("[4/5] Exportando tabelas CSV e gerando banco SQLite...")
    df_assets.to_csv(os.path.join(output_dir, "dim_empresa.csv"), index=False, encoding="utf-8-sig")
    df_weights.to_csv(os.path.join(output_dir, "fato_pesos_carteiras.csv"), index=False, encoding="utf-8-sig")
    df_resumo.to_csv(os.path.join(output_dir, "resumo_executivo_carteiras.csv"), index=False, encoding="utf-8-sig")
    df_monte_carlo.to_csv(os.path.join(output_dir, "fronteira_eficiente_monte_carlo.csv"), index=False, encoding="utf-8-sig")
    df_cotacoes_melted.to_csv(os.path.join(output_dir, "fato_cotacoes_diarias.csv"), index=False, encoding="utf-8-sig")

    # Popular SQLite
    db_path = os.path.join(output_dir, "fintech_esg.db")
    conn = sqlite3.connect(db_path)
    df_assets.to_sql("dim_empresa", conn, if_exists="replace", index=False)
    df_weights.to_sql("fato_pesos_carteiras", conn, if_exists="replace", index=False)
    df_resumo.to_sql("resumo_executivo_carteiras", conn, if_exists="replace", index=False)
    df_monte_carlo.to_sql("fronteira_eficiente_monte_carlo", conn, if_exists="replace", index=False)
    df_cotacoes_melted.to_sql("fato_cotacoes_diarias", conn, if_exists="replace", index=False)
    conn.close()

    print("[5/5] [OK] Case 3 concluido com dados reais da B3 e GHG Protocol em:", output_dir)
    return df_resumo, df_weights

if __name__ == "__main__":
    out_dir = r"C:\Users\lenovo\Desktop\projetoscopilot\cases gen\case-03-fintech-esg-portfolio\data"
    resumo, pesos = build_esg_fintech_project(out_dir)
    print("\n--- COMPARATIVO EXECUTIVO DE CARTEIRAS ---")
    print(resumo[["Carteira", "Retorno_Anualizado_%", "Volatilidade_Anualizada_%", "Indice_Sharpe", "Intensidade_Carbono_tCO2e_M", "Reducao_Carbono_vs_Tradicional_%"]])
    print("\n--- ALOCACAO DE PESOS POR ATIVO (%) ---")
    print(pesos[["Ticker", "Empresa", "Peso_Carteira_Tradicional_%", "Peso_Carteira_GreenAlpha_ESG_%", "Peso_Carteira_Min_Vol_%"]])

[1/5] Baixando cotacoes historicas reais da B3 via yfinance (2022-2025)...


/tmp/ipykernel_2676/1166225811.py:42: FutureWarning: YF.download() has changed argument auto_adjust default to True
  raw_data = yf.download(tickers, start="2022-01-01", end="2025-01-01", progress=False)['Close']


[2/5] Otimizando Carteiras (Markowitz Tradicional vs. Mínima Volatilidade vs. GreenAlpha ESG)...
[3/5] Simulando 2.500 carteiras de Monte Carlo para a Fronteira Eficiente...
[4/5] Exportando tabelas CSV e gerando banco SQLite...
[5/5] [OK] Case 3 concluido com dados reais da B3 e GHG Protocol em: C:\Users\lenovo\Desktop\projetoscopilot\cases gen\case-03-fintech-esg-portfolio\data

--- COMPARATIVO EXECUTIVO DE CARTEIRAS ---
                              Carteira  Retorno_Anualizado_%  \
0  Tradicional (Max Sharpe Irrestrito)                 27.66   
1        GreenAlpha ESG (Filtro Verde)                 25.92   
2      Minima Volatilidade (Defensiva)                 15.94   

   Volatilidade_Anualizada_%  Indice_Sharpe  Intensidade_Carbono_tCO2e_M  \
0                      18.55          0.925                        41.22   
1                      17.76          0.869                        20.00   
2                      14.65          0.371                        27.95   

   Reducao_